## Example 1: a line plot from a pandas DataFrame

In [ ]:
import holoviews as hv
import hvplot.pandas  # registers the .hvplot accessor on pandas objects
import numpy as np
import pandas as pd

df = pd.DataFrame({"x": np.arange(10), "y": np.arange(10) ** 2})
plot = df.hvplot.line(x="x", y="y")
print(f"plot type: {type(plot).__name__}, kdims={plot.kdims}, vdims={plot.vdims}")
assert isinstance(plot, hv.Curve)
assert [d.name for d in plot.kdims] == ["x"] and [d.name for d in plot.vdims] == ["y"]
assert plot.dimension_values("y").tolist() == [i * i for i in range(10)]
assert plot.range("y") == (0, 81)

# A Series works too
series_plot = pd.Series([3, 1, 2], name="v").hvplot.line()
assert isinstance(series_plot, hv.Curve)
assert series_plot.dimension_values("v").tolist() == [3, 1, 2]

## Example 2: scatter plots, one per group with `by`

In [ ]:
import holoviews as hv
import hvplot.pandas
import numpy as np
import pandas as pd

df = pd.DataFrame({"x": np.arange(10), "y": np.arange(10) * 2.0, "group": ["a", "b"] * 5})
scatter = df.hvplot.scatter(x="x", y="y")
print(f"scatter type: {type(scatter).__name__}, {len(scatter)} points")
assert isinstance(scatter, hv.Scatter) and len(scatter) == 10

by_group = df.hvplot.scatter(x="x", y="y", by="group")
print(f"grouped type: {type(by_group).__name__}, keys={list(by_group.keys())}")
assert isinstance(by_group, hv.NdOverlay)
assert sorted(by_group.keys()) == ["a", "b"]
assert by_group["a"].dimension_values("x").tolist() == [0, 2, 4, 6, 8]

## Example 3: a histogram

In [ ]:
import holoviews as hv
import hvplot.pandas
import numpy as np
import pandas as pd

df = pd.DataFrame({"y": np.arange(10) ** 2})
hist = df.hvplot.hist("y", bins=5)
counts = hist.dimension_values(1)
print(f"histogram type: {type(hist).__name__}, counts per bin: {counts.tolist()}")
assert isinstance(hist, hv.Histogram)
assert len(counts) == 5 and counts.sum() == 10
assert hist.edges[0] == 0 and hist.edges[-1] == 81

## Example 4: bar charts, tables and the hvPlot class

In [ ]:
import holoviews as hv
import hvplot.pandas
import pandas as pd
from hvplot import hvPlot

df = pd.DataFrame({"fruit": ["apple", "pear", "plum"], "count": [5, 3, 7]})
bars = df.hvplot.bar(x="fruit", y="count")
print(f"bars type: {type(bars).__name__}")
assert isinstance(bars, hv.Bars)
assert bars.dimension_values("fruit").tolist() == ["apple", "pear", "plum"]
assert bars.dimension_values("count").tolist() == [5, 3, 7]

table = df.hvplot.table(columns=["fruit", "count"])
assert isinstance(table, hv.Table) and len(table) == 3

# The accessor is a thin wrapper over the hvPlot class, which takes the data directly.
direct = hvPlot(df).bar(x="fruit", y="count")
assert isinstance(direct, hv.Bars)
assert direct.dimension_values("count").tolist() == [5, 3, 7]

## Example 5: render with Bokeh and save to HTML

In [ ]:
import os
import hvplot
import hvplot.pandas
import numpy as np
import pandas as pd

df = pd.DataFrame({"x": np.arange(10), "y": np.arange(10) ** 2})
plot = df.hvplot.line(x="x", y="y", title="Squares", xlabel="step", ylabel="value", width=400, height=250)

# render() turns the HoloViews object into a Bokeh figure carrying the plot options.
figure = hvplot.render(plot, backend="bokeh")
print(f"figure: {type(figure).__name__}, title={figure.title.text!r}, size={figure.width}x{figure.height}")
assert figure.title.text == "Squares"
assert figure.xaxis[0].axis_label == "step" and figure.yaxis[0].axis_label == "value"
assert (figure.width, figure.height) == (400, 250)

# save() writes a standalone HTML page.
hvplot.save(plot, "squares.html")
with open("squares.html", encoding="utf-8") as f:
    page = f.read()
print(f"squares.html: {os.path.getsize('squares.html')} bytes")
assert "<html" in page.lower() and "Squares" in page
assert callable(hvplot.help) and callable(hvplot.extension)